In [ ]:
import sys

print(sys.version)
print(sys.executable)


import torch

print(torch.__version__)
print(torch.backends.mps.is_available())  # True

device = torch.device("mps")

# Loading tokenizer


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

text = "Hello, how are you?"

tokens = tokenizer.encode(text)

for token in tokens:
    print(f"{token} ->> {tokenizer.decode([token])}")

print(f"vocab size: {tokenizer.vocab_size}")

# Loding base model


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

model_id = "Qwen/Qwen3-0.6B-Base"

path = hf_hub_download(repo_id=model_id, filename="model.safetensors")

weights = load_file(path, device="cpu")

print(list(weights.keys())[:10])

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

model.eval()

In [ ]:
print(model)  # Complete architecture
print(model.model.layers[0])  # First transformer block
weights = model.state_dict()  # All pretrained tensors
config = model.config  # Architecture hyperparameters

# Inference


In [ ]:
@torch.inference_mode()  # no backprop; dont track gradient
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()  # enables deterministic behavior

    for _ in range(max_new_tokens):
        # forward  pass: (batch, seq_len. vocab size)
        logits = model(token_ids).logits

        # gen last token's logits and select higesht-scoring token
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        # yield each token as soon as it generated. so you are streaming token
        yield next_token

        # append newly predicted token to the sequence
        token_ids = torch.cat([token_ids, next_token], dim=1)

## KV caching


In [ ]:
@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
):
    model.eval()

    # Process the entire prompt once (prefill)
    outputs = model(token_ids, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[:, -1, :]

    for _ in range(max_new_tokens):
        # Select the highest-scoring next token
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token

        # Process ONLY the new token, reusing cached K/V tensors
        outputs = model(
            next_token,
            past_key_values=cache,
            use_cache=True,
        )

        cache = outputs.past_key_values
        logits = outputs.logits[:, -1, :]

In [ ]:
prompt = "Explain large language models in a single sentence."

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt), device=device
).unsqueeze(0)

max_new_tokens = 100

In [ ]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    # with flush= True ,Immediately sends printed text to the output stream rather than waiting for buffering.
    print(tokenizer.decode(token[0]), end="", flush=True)

In [ ]:
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token[0]), end="", flush=True)

# Post-training lab — companion to Nathan Lambert’s RLHF Book

This is an **independent PyTorch learning notebook** that follows the subjects in [RLHF Book](https://github.com/natolambert/rlhf-book), rather than reproducing its training framework. Your original Qwen3-0.6B-Base loading, greedy decoding and KV-cache experiments remain above.

**Important:** This notebook imports the helpers in `rl_utils.py` (put the two files side by side). We will implement the *math* in small, inspectable pieces here, and graduate to the book’s reference training implementations as experiments get larger.

| Read in RLHF Book | Do in this notebook | Reference implementations |
|---|---|---|
| [Ch. 3 — Training overview](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/03-training-overview.md) | Logits → probabilities → sampling → log-prob scoring | [code/](https://github.com/natolambert/rlhf-book/tree/main/code) |
| [Ch. 4 — Instruction tuning](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/04-instruction-tuning.md) | Mask prompt, optimize completion NLL | [instruction_tuning/](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning) |
| [Ch. 5 — Reward models](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/05-reward-models.md) | Bradley–Terry preference loss | [reward_models/](https://github.com/natolambert/rlhf-book/tree/main/code/reward_models) |
| [Ch. 6 — Policy gradients](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/06-policy-gradients.md) | REINFORCE, PPO surrogate, GRPO advantages, KL | [policy_gradients/](https://github.com/natolambert/rlhf-book/tree/main/code/policy_gradients) |
| [Ch. 8 — Direct alignment](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/08-direct-alignment.md) | DPO preference loss | [direct_alignment/](https://github.com/natolambert/rlhf-book/tree/main/code/direct_alignment) |
| [Ch. 9 — Rejection sampling](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/09-rejection-sampling.md) | Best-of-N selection vs random baseline | [rejection_sampling/](https://github.com/natolambert/rlhf-book/tree/main/code/rejection_sampling) |
| [Ch. 12 — Synthetic data](https://github.com/natolambert/rlhf-book/blob/main/book/chapters/12-synthetic-data.md) | Reverse-KL distillation at student-sampled prefixes | [distillation/ (SDPO)](https://github.com/natolambert/rlhf-book/tree/main/code/distillation) |

**Model choice:** Keep **Qwen3-0.6B-Base** to study base-model behavior and SFT. Later, start RL rollouts from an **instruction-tuned** policy so it can follow task instructions and stop reliably. The book’s runnable GRPO example uses Qwen3-1.7B, which is larger than this lightweight local notebook.


## 1. Temperature, top-k, top-p, entropy, and token sampling

- **Temperature** $T$: divide logits by $T$; a value of `0` selects the highest-logit token deterministically.
- **Top-k**: keep exactly the `k` highest-logit tokens.
- **Top-p**: keep the *smallest* descending-probability prefix whose total reaches at least `p` (including the token that crosses the threshold).
- **Entropy**: $-\sum_i p_i\log p_i$ measures policy uncertainty in **nats**.

The helper works for batch-shaped logits `[B, vocab_size]` and returns sampled IDs plus both flavors of log-probability.

In [ ]:
# All reusable implementations live in the adjacent rl_utils.py file.
# Keep this notebook and rl_utils.py in the SAME working directory.
from rl_utils import (
    SamplingConfig,
    sampling_logits,
    categorical_entropy,
    sample_next_token,
    rollout_with_logprobs,
    completion_logprobs,
    sequence_logprob,
    tokenwise_kl,
    masked_mean,
    reinforce_loss,
    ppo_clipped_loss,
)
import torch
import torch.nn.functional as F

print("Loaded utilities from:", __import__("rl_utils").__file__)


In [ ]:
# Fast sanity check without running Qwen
raw_logits = torch.tensor([[3.0, 2.0, 1.0, 0.0]], device=device)

for config in [
    SamplingConfig(temperature=0),
    SamplingConfig(temperature=0.7),
    SamplingConfig(temperature=1.0, top_k=2),
    SamplingConfig(temperature=1.0, top_p=0.8),
    SamplingConfig(temperature=0.8, top_k=3, top_p=0.9),
]:
    filtered = sampling_logits(raw_logits, config)
    sample = sample_next_token(raw_logits, config)
    print(
        f"{config} | eligible={torch.isfinite(filtered).sum().item()} "
        f"| token={sample['token_ids'].item()} "
        f"| raw logp={sample['model_logprob'].item():.3f} "
        f"| sampling logp={sample['behavior_logprob'].item():.3f}"
    )

## 2. KV-cached rollout + teacher-forced log-prob scoring

A rollout samples tokens with **no gradients** and stores old-policy scores. When training, run a **fresh teacher-forced model forward pass with gradients** on the sampled completion.

For a prompt of length `P` and a completion of length `C`, the logits at positions `P-1` through `P+C-2` predict exactly the `C` completion tokens. This one-token shift is easy to get wrong. The rollout tracks **EOS** as an action if sampled; unlike a text streaming function, it does not silently drop EOS from the scores.

These minimal utilities assume **one unpadded example for generation** and **equal-length, unpadded prompts when scoring a batch**. Add padding/attention masks before scaling up.

In [ ]:
@torch.no_grad()
def rollout_with_logprobs(
    model,
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    config: SamplingConfig,
    eos_token_id: int | None = None,
) -> dict[str, torch.Tensor]:
    """KV-cached single-example rollout; includes EOS if sampled.

    Returns completion_ids, model_logprobs, behavior_logprobs [1, T].
    Saving logprobs here is for tracking the behavior/old policy; to train,
    re-run completion_logprobs WITHOUT no_grad to obtain fresh gradients.
    """
    if prompt_ids.ndim != 2 or prompt_ids.size(0) != 1 or prompt_ids.size(1) == 0:
        raise ValueError("prompt_ids must have shape [1, prompt_length>=1]")
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be >= 1")

    model.eval()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]

    tokens, model_lps, behavior_lps, entropies = [], [], [], []
    for step in range(max_new_tokens):
        sample = sample_next_token(next_logits, config)
        next_token = sample["token_ids"]
        tokens.append(next_token)
        model_lps.append(sample["model_logprob"])
        behavior_lps.append(sample["behavior_logprob"])
        entropies.append(sample["behavior_entropy"].unsqueeze(1))

        if eos_token_id is not None and bool((next_token == eos_token_id).all().item()):
            break
        if step < max_new_tokens - 1:
            outputs = model(next_token, past_key_values=cache, use_cache=True)
            cache = outputs.past_key_values
            next_logits = outputs.logits[:, -1, :]

    return {
        "completion_ids": torch.cat(tokens, dim=1),
        "model_logprobs": torch.cat(model_lps, dim=1),
        "behavior_logprobs": torch.cat(behavior_lps, dim=1),
        "behavior_entropy": torch.cat(entropies, dim=1),
    }


def completion_logprobs(
    model,
    prompt_ids: torch.Tensor,
    completion_ids: torch.Tensor,
) -> torch.Tensor:
    """Teacher-forced, differentiable model log p for completion tokens.

    Prompt: [B, P], completion: [B, C]. Returns [B, C].
    Shift alignment: logits at position P-1 predict the FIRST completion token.
    No padding or variable-length prompts in this minimal implementation.
    """
    if (prompt_ids.ndim != 2 or completion_ids.ndim != 2 or
        prompt_ids.size(0) != completion_ids.size(0) or
        prompt_ids.size(1) == 0 or completion_ids.size(1) == 0):
        raise ValueError("Use [B, P>=1] prompts and [B, C>=1] completions")

    input_ids = torch.cat([prompt_ids, completion_ids], dim=-1)
    logits = model(input_ids, use_cache=False).logits
    start = prompt_ids.size(1) - 1
    end = start + completion_ids.size(1)
    completion_logits = logits[:, start:end, :].float()
    return F.log_softmax(completion_logits, dim=-1).gather(
        -1, completion_ids.unsqueeze(-1)
    ).squeeze(-1)


def sequence_logprob(token_logprobs: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
    """Sum of chosen-token log probabilities, per example [B]."""
    if mask is None:
        return token_logprobs.sum(dim=-1)
    return (token_logprobs * mask).sum(dim=-1)




In [ ]:
# Try sampling from your already-loaded Qwen model.
# These values are useful for creative generation, but for strict on-policy
# training of the RAW model start with (1.0, None, 1.0).
config = SamplingConfig(temperature=0.8, top_k=50, top_p=0.95)

rollout = rollout_with_logprobs(
    model=model,
    prompt_ids=input_token_ids_tensor,
    max_new_tokens=24,
    config=config,
    eos_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(rollout["completion_ids"][0], skip_special_tokens=True))
print("Token count:", rollout["completion_ids"].shape[1])
print("Raw model sequence logp:", sequence_logprob(rollout["model_logprobs"]).item())
print("Sampling behavior sequence logp:", sequence_logprob(rollout["behavior_logprobs"]).item())

# Verify that teacher-forced scoring and cached-rollout raw logp agree
# (small numerical differences are possible from cache/precision).
with torch.no_grad():
    rescored = completion_logprobs(model, input_token_ids_tensor, rollout["completion_ids"])
print("Rescore agreement:", torch.allclose(rescored, rollout["model_logprobs"], atol=1e-2, rtol=1e-2))

## 3. Distillation losses + RL policy losses

**At a fixed prefix**, forward KL is $D_{KL}(p_{teacher}\|p_{student})$ and reverse KL is $D_{KL}(p_{student}\|p_{teacher})$. Below both are computed across **the full vocabulary**, not just the sampled token. `teacher_logits` are detached; the student keeps gradients.

**On-policy distillation** adds another dimension: the prefixes themselves are visited by the student policy, rather than always taken from a fixed teacher dataset. Computing reverse KL at teacher-provided prefixes alone does **not** make a procedure on-policy.

For RL, `reinforce_loss` applies a scalar advantage per completion. `ppo_clipped_loss` implements only the **tokenwise clipped policy surrogate**; a real PPO trainer also needs appropriate rewards/advantages, frozen old-policy scores, batching/masks, optimization, and often a reference-policy KL penalty.

In [ ]:
def tokenwise_kl(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    direction: str = "forward",
) -> torch.Tensor:
    """Per-position KL over the entire vocab; shape [B, T].

    forward: KL(teacher || student), akin to soft-label cross-entropy.
    reverse: KL(student || teacher), at the SAME fixed prefixes.
    Teacher is detached, student retains gradient.
    """
    student_lp = F.log_softmax(student_logits.float(), dim=-1)
    teacher_lp = F.log_softmax(teacher_logits.detach().float(), dim=-1)
    if direction == "forward":
        return (teacher_lp.exp() * (teacher_lp - student_lp)).sum(dim=-1)
    if direction == "reverse":
        return (student_lp.exp() * (student_lp - teacher_lp)).sum(dim=-1)
    raise ValueError("direction must be 'forward' or 'reverse'")


def masked_mean(losses: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
    """Average over valid tokens, or over all entries when mask is absent."""
    if mask is None:
        return losses.mean()
    return (losses * mask).sum() / mask.sum().clamp_min(1)


def reinforce_loss(
    new_logprobs: torch.Tensor,
    advantages: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """Sequence-level REINFORCE surrogate, with stop-grad advantages [B]."""
    if advantages.ndim != 1:
        raise ValueError("advantages must have shape [B]")
    return -(sequence_logprob(new_logprobs, mask) * advantages.detach()).mean()


def ppo_clipped_loss(
    new_logprobs: torch.Tensor,
    old_logprobs: torch.Tensor,
    advantages: torch.Tensor,
    mask: torch.Tensor | None = None,
    clip_eps: float = 0.2,
) -> torch.Tensor:
    """Minimal tokenwise PPO policy surrogate (no value or reference KL loss).

    new/old: [B,T], advantages: [B] or [B,T], mask: optional [B,T].
    This assumes old/new logprobs refer to the SAME policy definition.
    """
    if advantages.ndim == 1:
        advantages = advantages.unsqueeze(1)
    ratio = (new_logprobs - old_logprobs.detach()).exp()
    advantage = advantages.detach()
    unclipped = ratio * advantage
    clipped = ratio.clamp(1 - clip_eps, 1 + clip_eps) * advantage
    return masked_mean(-torch.minimum(unclipped, clipped), mask)


In [ ]:
# Distillation sanity check with a tiny invented vocabulary.
# Replace these with logits from your student/teacher models later.
student_logits = torch.tensor([[[2.0, 0.3, -0.4]]], requires_grad=True)
teacher_logits = torch.tensor([[[0.1, 2.0, -0.5]]])

forward_kl = tokenwise_kl(student_logits, teacher_logits, "forward").mean()
reverse_kl = tokenwise_kl(student_logits, teacher_logits, "reverse").mean()
print(f"KL(teacher || student): {forward_kl.item():.4f}")
print(f"KL(student || teacher): {reverse_kl.item():.4f}")

# Either loss has a differentiable gradient for student logits.
reverse_kl.backward()
print("Student gradient:", student_logits.grad)

## 4. Minimal policy-gradient / PPO preparation

For a raw-policy on-policy rollout, keep temperature `1`, top-k disabled and top-p `1`. Store `old_logprobs` without gradients. After you receive a reward and compute an **advantage**, score the same completion again through the trainable student.

The snippet intentionally **does not call `optimizer.step()`**: this is a foundation, not a full RL trainer. It also does not include a value model or KL regularizer. Do **not** use the filtered-sampler `behavior_logprobs` interchangeably with raw model log-probs in a PPO ratio.

In [ ]:
# Small real-model example; uses the model already loaded above.
raw_policy = SamplingConfig(temperature=1.0, top_k=None, top_p=1.0)
training_rollout = rollout_with_logprobs(
    model, input_token_ids_tensor, max_new_tokens=8,
    config=raw_policy, eos_token_id=tokenizer.eos_token_id,
)

# This forward pass has gradients (unlike rollout_with_logprobs).
model.train()
new_logprobs = completion_logprobs(model, input_token_ids_tensor, training_rollout["completion_ids"])
old_logprobs = training_rollout["model_logprobs"].detach()

# Demonstration advantage, NOT a learned reward or actual RL signal.
advantages = torch.tensor([1.0], device=new_logprobs.device)
pg_loss = reinforce_loss(new_logprobs, advantages)
ppo_loss = ppo_clipped_loss(new_logprobs, old_logprobs, advantages, clip_eps=0.2)
print("REINFORCE surrogate:", pg_loss.item())
print("PPO policy surrogate:", ppo_loss.item())

# In a real training step: optimizer.zero_grad(); ppo_loss.backward(); optimizer.step()
# Repeating PPO steps requires frozen old_logprobs and recomputed new_logprobs.
model.eval()

### Distillation on completion prefixes (template)

To compute **token-level KL at the completion positions**, run student and teacher over the *same* `prompt + completion`, take positions `P-1 : P+C-1`, then call `tokenwise_kl`. This is useful for supervised-prefix distillation and for on-policy distillation **when the completions/prefixes were sampled from the student**.

```python
full_ids = torch.cat([prompt_ids, completion_ids], dim=1)
start = prompt_ids.size(1) - 1
end = start + completion_ids.size(1)

student_logits = student_model(full_ids, use_cache=False).logits[:, start:end, :]
with torch.no_grad():
    teacher_logits = teacher_model(full_ids, use_cache=False).logits[:, start:end, :]

loss = tokenwise_kl(student_logits, teacher_logits, "reverse").mean()
loss.backward()  # gradient flows only into the student
```

**Important:** `log p(completion)` is the *sum* of conditional token log-probs, not the same thing as greedy decoding. You can score a fixed completion this way without it ever being the model's argmax sequence. Full-vocabulary KL is different from comparing the two models' log-probs on just one sampled completion; sampled-token estimates need careful treatment of the sampling distribution and gradients.

All decoding, scoring and loss helpers used above **come from** `rl_utils.py`. The notebook deliberately spells out the math in the experiments below.

## Book Chapter 4 — supervised fine-tuning (SFT)

For prompt $x$ and response tokens $y_1,\dots,y_T$, optimize

$$\mathcal L_{\rm SFT}=-\frac{1}{T}\sum_{t=1}^{T}\log\pi_\theta(y_t\mid x,y_{<t}).$$

The **prompt** provides context but contributes **zero loss**. `completion_logprobs` automatically aligns the logits with the completion tokens, so no manual shift or `-100` labels are needed for this unpadded single-example demonstration.

The code below computes loss on one teacher-forced response; it does **not** update the model.

In [ ]:
# SFT mini-lab (Chapter 4). This is a forward pass, not a full trainer.
sft_prompt = torch.tensor(tokenizer.encode("Question: What is 2 + 2?\nAnswer:"), device=device).unsqueeze(0)
sft_answer = torch.tensor(tokenizer.encode(" 4"), device=device).unsqueeze(0)

model.train()
sft_logprobs = completion_logprobs(model, sft_prompt, sft_answer)
sft_loss = -sft_logprobs.mean()
print("completion tokens:", sft_answer.tolist())
print("completion log-probs:", sft_logprobs.detach().tolist())
print("SFT NLL:", sft_loss.item())
# For training: optimizer.zero_grad(); sft_loss.backward(); optimizer.step()
model.eval()


## Book Chapter 5 — reward models (Bradley–Terry)

A preference reward model assigns scalar scores $r(x,y)$. If human feedback says $y^+$ is preferred over $y^-$, a standard loss is

$$\mathcal L_{\rm RM}=-\log\sigma(r(x,y^+)-r(x,y^-)).$$

The following uses *toy reward scores* to demonstrate the objective; it's not fitting a reward head.

In [ ]:
# Toy chosen/rejected reward scores (differentiable).
chosen_reward = torch.tensor([1.4, 0.3], requires_grad=True)
rejected_reward = torch.tensor([0.5, 0.8], requires_grad=True)
reward_model_loss = -F.logsigmoid(chosen_reward - rejected_reward).mean()
print("Bradley-Terry preference loss:", reward_model_loss.item())
print("P(chosen preferred):", torch.sigmoid(chosen_reward - rejected_reward).tolist())


## Book Chapter 6 — GRPO-style group advantages

For a prompt, sample $G$ completions and evaluate each with a verifier/reward. A common GRPO construction uses a **relative advantage** per completion:

$$A_i=\frac{r_i-\bar r}{\operatorname{std}(r)+\epsilon}.$$

The policy loss then weights each completion's token log-prob ratios by $A_i$, with optional clipping and KL penalties. Our `ppo_clipped_loss` is **only** the clipped policy component — it is not a complete GRPO implementation. The book's reference code also adds proper rollout batching, masks, reward functions, and configurable estimators.

In [ ]:
# Four completions of ONE prompt: each one gets a reward.
# Here we use toy log-probs rather than running 4 Qwen rollouts.
group_rewards = torch.tensor([1.0, 0.0, 1.0, 0.0])
group_advantages = (group_rewards - group_rewards.mean()) / group_rewards.std(unbiased=False).clamp_min(1e-8)

# Two action tokens per sampled completion; values are illustrative.
old_lp = torch.tensor([[-0.6, -0.8], [-1.1, -1.2], [-0.7, -0.7], [-1.0, -1.3]])
new_lp = old_lp.clone().requires_grad_()
group_policy_loss = ppo_clipped_loss(new_lp, old_lp, group_advantages, clip_eps=0.2)
group_policy_loss.backward()
print("Rewards:", group_rewards.tolist())
print("Relative advantages:", group_advantages.tolist())
print("Clipped group policy objective:", group_policy_loss.item())
print("Gradient for new logprobs:", new_lp.grad.tolist())

# IMPORTANT: For real training, new_lp must come from the trainable model,
# old_lp from a frozen rollout snapshot, and reward from a verifier/RM.


## Book Chapter 8 — Direct Preference Optimization (DPO)

DPO compares **sequence log-prob differences** for chosen and rejected responses under the current policy and a frozen reference policy:

$$\mathcal L_{\rm DPO}=-\log\sigma\big(\beta\big[(\log\pi_\theta(y^+|x)-\log\pi_\theta(y^-|x))-(\log\pi_{\rm ref}(y^+|x)-\log\pi_{\rm ref}(y^-|x))\big]\big).$$

These are **model log-probs of fixed completions** (not greedy decoding, and not necessarily the truncated sampling distribution). For real DPO, calculate them by teacher forcing with `completion_logprobs`, and sum the chosen-token log-probs over response tokens.

In [ ]:
beta = 0.1
# Illustrative sequence log-probabilities, chosen vs rejected.
policy_chosen = torch.tensor([-8.0, -5.0], requires_grad=True)
policy_rejected = torch.tensor([-10.0, -4.0], requires_grad=True)
ref_chosen = torch.tensor([-9.0, -5.0])
ref_rejected = torch.tensor([-9.5, -4.5])

policy_margin = policy_chosen - policy_rejected
reference_margin = ref_chosen - ref_rejected
dpo_loss = -F.logsigmoid(beta * (policy_margin - reference_margin)).mean()
print("policy margins:", policy_margin.detach().tolist())
print("reference margins:", reference_margin.tolist())
print("DPO loss:", dpo_loss.item())


## Book Chapter 9 — rejection sampling / best-of-N

Generate $N$ completions with one policy, score each using a verifier or reward model, and train on selected completions using SFT. **Best-of-N is not the same thing as top-p or top-k token sampling.** You select among *complete responses*, not individual next-token candidates.

Always compare to a **randomly selected** rollout from the same pool, as the book's reference experiments do.

In [ ]:
# In practice, replace these example scores with verifier or reward-model scores.
candidates = ["3", "4", "5", "4, because 2+2=4"]
scores = torch.tensor([0.0, 1.0, 0.0, 1.0])

best_idx = int(scores.argmax())
random_idx = int(torch.randint(len(candidates), (1,)).item())
print("Best-of-N:", candidates[best_idx], "reward:", scores[best_idx].item())
print("Random baseline:", candidates[random_idx], "reward:", scores[random_idx].item())
# A real pipeline: save selected prompt/completion pairs and then SFT on them.


## Book Chapter 12 — on-policy distillation / SDPO bridge

The usual token-level **reverse KL** is $D_{KL}(\pi_{student}(\cdot\mid s)\|\pi_{teacher}(\cdot\mid s))$. It compares **distributions over all next tokens at the same state/prefix**. For on-policy distillation, sample the **prefixes** from the student.

[RLHF Book's SDPO example](https://github.com/natolambert/rlhf-book/tree/main/code/distillation) is more specific: it samples groups of student solutions, verifies them, and uses a *correct sibling* as privileged context for a **self-teacher**. Its code approximates reverse KL with student top-K tokens **plus a tail bucket**. Our `tokenwise_kl` uses the full vocabulary on toy/small examples; it's conceptually related, but not the full SDPO algorithm.

The next practical experiment is to choose a verifiable task (e.g. reverse a short word), generate multiple solutions with an instruct policy, score exact correctness, and compare GRPO against an on-policy teacher-distillation run.

In [ ]:
# Full-distribution KL at fixed prefixes: all student gradients are retained.
student_logits = torch.tensor([[[1.0, 2.0, 0.0]]], requires_grad=True)
teacher_logits = torch.tensor([[[0.2, 2.4, -0.1]]])

loss_forward = tokenwise_kl(student_logits, teacher_logits, "forward").mean()
loss_reverse = tokenwise_kl(student_logits, teacher_logits, "reverse").mean()
print("forward KL (teacher || student):", loss_forward.item())
print("reverse KL (student || teacher):", loss_reverse.item())

# This calculation alone is *not* on-policy. The preceding prefixes must
# have been generated by the student to give it that interpretation.


### What to implement next, in the book's order

1. **Chapter 4 — SFT:** build prompt masking + a minimal `AdamW` loop on a tiny task; validate that the base model learns instruction formatting.
2. **Chapter 5 — reward:** implement an exact-answer verifier, then a tiny pairwise reward head with Bradley–Terry loss.
3. **Chapter 6 — RL:** collect 4–8 rollouts per prompt; begin with REINFORCE, add group-normalized advantages, clipped GRPO ratios, and optional reference KL.
4. **Chapters 8–9:** compare DPO on chosen/rejected answers against best-of-N SFT with a matched random baseline.
5. **Chapter 12:** sample student states, compare forward/reverse KL, then read the book's `distillation/loss.py` to implement top-K + tail and SDPO demonstrations.

**Notebook scope:** This is a notebook of tested building blocks and illustrative objectives, not a turnkey distributed RLHF trainer. For full experiments use the repo's `code/` implementations; its current training scripts target CUDA hardware, so an Apple M1 Pro may need smaller models, batches or custom MPS adjustments.


## torch.compile


In [ ]:
compiled_model = torch.compile(
    model,
    backend="inductor",
    mode="default",
)

In [ ]:
with torch.inference_mode():
    logits = compiled_model(
        input_token_ids_tensor,
        use_cache=False,
    ).logits

print(logits.shape)

In [ ]:
import time


def benchmark(model, n=100):
    # Warmup
    with torch.inference_mode():
        for _ in range(3):
            model(input_token_ids_tensor, use_cache=False)

    torch.mps.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(n):
            model(input_token_ids_tensor, use_cache=False)

    torch.mps.synchronize()

    return (time.perf_counter() - start) / n


eager_time = benchmark(model)
compiled_time = benchmark(compiled_model)

print(f"Eager:    {eager_time * 1000:.2f} ms")
print(f"Compiled: {compiled_time * 1000:.2f} ms")
print(f"Speedup:  {eager_time / compiled_time:.2f}x")